In [2]:
!pip install -q -U google-genai pypdf

In [6]:
from google import genai
from google.colab import userdata
from google.genai import types
import numpy as np

client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))
EMB_MODEL="gemini-embedding-001"
EMB_DIM=768
MODEL="gemini-3.5-flash-lite"

In [11]:
#1.Prepare the document
#1.1.Upload PDF
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(f"PDF:{pdf_name} UPLOADED")

Saving AI_Job_Suitability_Dataset.pdf to AI_Job_Suitability_Dataset (2).pdf
PDF:AI_Job_Suitability_Dataset (2).pdf UPLOADED


In [14]:
#1.2 Extract text from uploaded pdf
from pypdf import PdfReader
reader=PdfReader(pdf_name)
print("NUMBER OF PAGES: ",len(reader.pages))
text=""
for page in reader.pages:
  text+=page.extract_text()+"\n"
print("TEXT EXTRACTED WHERE NUMBER OF CHARACTER IS: ",len(text))

NUMBER OF PAGES:  3
TEXT EXTRACTED WHERE NUMBER OF CHARACTER IS:  4696


In [15]:
#1.3 Chunking (Overlapping chunking method)
def chunk_text(text,chunk_size=800,overlap=20):
  start=0
  chunks=[]
  while start<len(text):
    end=start+chunk_size
    chunks.append(text[start:end])
    start=end-overlap
  return chunks

chunks=chunk_text(text)
print("NUMBER OF CHUNKS GENERATED: ",len(chunks))


NUMBER OF CHUNKS GENERATED:  7


In [17]:
#Embed each chunk
def embed_chunk(chunk):
  response=client.models.embed_content(
      model=EMB_MODEL,
      contents=chunk,
      config=types.EmbedContentConfig(
          output_dimensionality=EMB_DIM
      )
  )
  return response.embeddings[0].values

chunk_embeddings=[]
for chunk in chunks:
  chunk_embeddings.append(embed_chunk((chunk)))
print("Embedding Done!")
#convert into array
chunk_embeddings=np.array(chunk_embeddings)
print(chunk_embeddings.shape)

Embedding Done!
(7, 768)


In [18]:
#Find similar Job
def cosine_sim(a,b):
  a=np.array(a)
  b=np.array(b)

  return (float((np.dot(a,b))/(np.linalg.norm(a))*(np.linalg.norm(b))))


In [20]:
def retrieve(info,k=3):
  similarity=[]
  info_embed=embed_chunk(info)
  for i,chunk in enumerate(chunk_embeddings):
    score=cosine_sim(info_embed,chunk)
    similarity.append((i,score))

  similarity.sort(key=lambda x:x[1],reverse=True)
  similarity=similarity[:k]
  return similarity

In [26]:
system_instruction=f"""
                  Act as Senior Placement Officer in placement cell.
                  Now you will be provided with processed pdf data of available jobs and student info with skills.
                  Based on available job information and student info skills and interested job role suggest him jobs that he can try and give interview.
                  Provide result in structured format where the details must contain infomation of job job discription then tell why does he suit for that job and suggest preperation before placement drive for that job.
                  Dont not provide any assumed/wrong data at any point of time,if data is not available reply data is not available.
                  every starting of new session reply with Hi i am you placement assistant,lets crack placement together.
                  provide response in structure format in points with highlighted points.
                  if student info is provided is having very less info then tell user to provide with more info
                  Also print at last of every job is student eligible(cgpa age semester attandance as mentioned in pdf data) in all cases, if not eligible in 1or2cases specify them
                  give  detailed analysis
                  answer everthing from pdf only formating and based data from pdf u can generate answers


"""
chats=client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        temperature=0.6,
        max_output_tokens=1500,
        system_instruction=system_instruction,
        thinking_config=types.ThinkingConfig(thinking_level='low')
    ),
    history=[]
)


print("Enter student info like name usn semester age attandance cgpa skills role interested\nEnter quit,bye,exit to terminate session")
while True:
  info_input=input("STUDENT: ")
  if info_input.lower() in ['quit','exit','bye']:
    print("Thank you for using our service")
    break
  similarity=retrieve(info_input)
  sim_jobs=[]
  for index,score in similarity:
    sim_jobs.append(chunks[index])

  prompt=f"""
        By using the student information is provided: {info_input}.
        then also similar job roles are found from pdf and processed data is provided: {sim_jobs}
        by using these two sources provided student with best 3 job roles that he can provide interview and have high chance of selection
      """
  response=chats.send_message(prompt)
  print(f"PLACEMENT ASSITANT:{response.text}")

Enter student info like name usn semester age attandance cgpa skills role interested
Enter quit,bye,exit to terminate session
STUDENT: Rahul Sharma, USN 1BM23CS001, Semester 6, Age 21, Attendance 82%, CGPA 8.2, Skills: Python, SQL, Machine Learning, NumPy, Pandas, Git, Interested Role: AI/ML, Generative AI, Data Analyst
PLACEMENT ASSITANT:Hi i am you placement assistant,lets crack placement together.

Based on your academic profile, skills, and areas of interest, here is the detailed placement analysis and top job recommendations derived strictly from the provided placement data.

---

### **1. Generative AI Intern — AIWorks Research**

*   **Job Description:** Build LLM-powered applications using prompt engineering, embeddings, retrieval augmented generation and APIs.
*   **Why You Suit This Job:** Your profile explicitly matches the required core competencies. You have proficiency in **Python**, which is essential for this role, and your stated interest in **Generative AI** and **AI/

In [5]:
#printing models name
for m in client.models.list():
  print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-2-preview
models/gemin